# INT8 Quantization Bug — Arduino Nicla Vision / OpenMV v4.8.1

## Bug Summary
Any INT8 or UINT8 fully-quantized TFLite model fails to load on the Arduino Nicla Vision running OpenMV v4.8.1 with the error:

```
Failed to allocate tensors
```

A float model of **identical architecture and similar size** loads and runs correctly on the same hardware.

## Hardware & Firmware
- **Board:** Arduino Nicla Vision
- **Firmware:** OpenMV v4.8.1 / MicroPython v1.26.0-77 / STM32H747

## What This Notebook Does
1. Builds the same model architecture used in this project
2. Trains on dummy data (no dataset upload needed — the bug occurs at load time, not inference)
3. Exports both an INT8 and a Float TFLite model
4. Confirms INT8 dtypes are correct in TFLite before flashing
5. Provides the Nicla inference script to reproduce the failure on-device

## Step 1: Install Dependencies

In [ ]:
!pip install tensorflow --upgrade

## Step 2: Build and Train the Model

This is the same architecture used in the Nicla Vision Gesture Recognition project:
- Input: 32×32 grayscale image
- Output: 3 classes (LEFT, RIGHT, NONE)

Dummy data is used here — the bug occurs at model **load time** on the Nicla, not during inference, so real gesture data is not required to reproduce it.

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

def build_model(input_shape=(32, 32, 1), num_classes=3):
    # Note: input_shape is (32, 32, 1) — images are captured at 64x64
    # and downscaled to 32x32 during data capture on the Nicla.
    model = models.Sequential([
        layers.Conv2D(2, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(4, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 225 dummy samples — 75 per class, matching real dataset size
np.random.seed(42)
X_dummy = np.random.rand(225, 32, 32, 1).astype(np.float32)
y_dummy = np.array([0, 1, 2] * 75)

model.fit(X_dummy, y_dummy, epochs=5, batch_size=16, verbose=1)

## Step 3: Export SavedModel

In [ ]:
model.export('/tmp/gesture_saved_model')
print("SavedModel exported.")

## Step 4: Convert to INT8

This produces the model that **fails** on the Nicla with `Failed to allocate tensors`.

Note: The conversion succeeds here in Colab and the dtype check below confirms it is correctly quantized. The failure only occurs when the model is loaded on-device via `ml.Model()`.

In [ ]:
def representative_dataset():
    for i in range(len(X_dummy)):
        yield [X_dummy[i:i+1]]

converter = tf.lite.TFLiteConverter.from_saved_model('/tmp/gesture_saved_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_int8 = converter.convert()

with open("gesture_int8.tflite", "wb") as f:
    f.write(tflite_int8)

print("INT8 model size: %.2f KB" % (len(tflite_int8) / 1024))

# Confirm dtypes — both should be int8
interpreter = tf.lite.Interpreter(model_content=tflite_int8)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input  dtype:", input_details[0]['dtype'])   # expected: int8
print("Output dtype:", output_details[0]['dtype'])  # expected: int8
print("Input  shape:", input_details[0]['shape'])

## Step 5: Convert to Float

This produces the model that **works** on the Nicla. Identical architecture, no quantization.

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model('/tmp/gesture_saved_model')
tflite_float = converter.convert()

with open("gesture_float.tflite", "wb") as f:
    f.write(tflite_float)

print("Float model size: %.2f KB" % (len(tflite_float) / 1024))

## Step 6: Download Both Models

Download both `.tflite` files to your machine, then copy them to `/flash` on the Nicla using OpenMV IDE.

In [ ]:
from google.colab import files
files.download("gesture_int8.tflite")
files.download("gesture_float.tflite")

## Step 7: Reproduce the Bug on the Nicla

1. Copy `gesture_int8.tflite` to `/flash` on the Nicla using OpenMV IDE
2. Copy the script below into OpenMV IDE as `main.py`
3. Run it — you should see:
```
Traceback (most recent call last):
  File "main.py", line 12, in <module>
OSError: Failed to allocate tensors
```
4. To confirm the float model works, change the model path to `gesture_float.tflite` and rerun — it should load and run without error.

### Nicla `main.py`

```python
import sensor, image, time, ml
from ulab import numpy as np

sensor.reset()
sensor.set_pixformat(sensor.GRAYSCALE)
sensor.set_framesize(sensor.B64X64)
sensor.skip_frames(time=2000)

# This line raises: OSError: Failed to allocate tensors
model = ml.Model("/flash/gesture_int8.tflite")

# Workaround — swap in the float model:
# model = ml.Model("/flash/gesture_float.tflite")

LABELS = ["LEFT", "RIGHT", "NONE"]

while True:
    img = sensor.snapshot()
    img_resized = img.copy(x_scale=0.5, y_scale=0.5)
    raw = model.predict([img_resized])
    scores = raw[0][0]
    best_idx = int(np.argmax(scores))
    print("Gesture:", LABELS[best_idx])
    time.sleep_ms(500)
```